# Modelo de temperatura — Método de Euler "paso a paso"

Este cuaderno resuelve la Ley de enfriamiento de Newton **sin usar librerías de integración**: se programa el método de Euler directamente con arreglos de NumPy, igual que se hace a mano.

$$\frac{dT}{dt} = -k\,(T - T_a)$$

Método de Euler: $T_{n+1} = T_n + h\cdot\bigl(-k\,(T_n - T_a)\bigr)$

![Modelo de temperatura](resources/Modelo-temperatura.png)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

* Parámetros (todo el cuaderno lee los valores de esta celda)

In [ ]:
# Parámetros del modelo
k     = 0.15    # constante de transferencia de calor (1/min)
T_amb = 20.0    # temperatura del ambiente (°C)

# Tamaño del paso
h = 0.1

# Tiempo final
t_final = 30

# Listas para las variaciones
k_lista  = [0.05, 0.1, 0.2, 0.4, 0.8]
Ta_lista = [0, 10, 20, 30, 50]
T0_lista = [-10, 5, 20, 50, 90]

# Prueba de estabilidad de Euler
k_rapido = 0.6
h_lista  = [0.25, 1.0, 2.5, 3.6]

# Límites del eje de temperatura en el campo de direcciones
T_min_graf, T_max_graf = -20, 100

* Condiciones iniciales

In [ ]:
T_0 = 90.0     # objeto caliente

In [ ]:
T_2_0 = 5.0    # objeto frío

* Parámetros del método

In [ ]:
t = np.arange(0, t_final + h, h)

* Vectores para almacenar la solución

In [ ]:
T = np.zeros(len(t))
T[0] = T_0

In [ ]:
T_2 = np.zeros(len(t))
T_2[0] = T_2_0

* Método de Euler

In [ ]:
for n in range(len(t) - 1):

    # Derivada en el punto actual
    dT = -k * (T[n] - T_amb)

    # Siguiente punto
    T[n+1] = T[n] + h * dT

In [ ]:
for n in range(len(t) - 1):

    # Derivada en el punto actual
    dT_2 = -k * (T_2[n] - T_amb)

    # Siguiente punto
    T_2[n+1] = T_2[n] + h * dT_2

* Gráficas T(t)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(t, T, label="Objeto caliente T(t)")
plt.plot(t, T_2, label="Objeto frío T(t)")
plt.axhline(T_amb, color="gray", linestyle="--", label="Temperatura ambiente")

plt.xlabel("t (min)")
plt.ylabel("Temperatura (°C)")
plt.title("Modelo de temperatura — Método de Euler")
plt.legend()
plt.grid()

plt.show()

* Comparación con la solución exacta  $T(t) = T_a + (T_0 - T_a)e^{-kt}$

In [ ]:
T_exacta = T_amb + (T_0 - T_amb) * np.exp(-k * t)
error = np.abs(T - T_exacta)

fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))

axs[0].plot(t, T_exacta, "k", linewidth=2, label="Exacta")
axs[0].plot(t, T, "--", label=f"Euler (h = {h})")
axs[0].set_xlabel("t (min)")
axs[0].set_ylabel("Temperatura (°C)")
axs[0].set_title("Euler vs. exacta")
axs[0].legend()
axs[0].grid()

axs[1].plot(t, error, color="tab:red")
axs[1].set_xlabel("t (min)")
axs[1].set_ylabel("|error| (°C)")
axs[1].set_title(f"Error absoluto (máximo = {error.max():.4f} °C)")
axs[1].grid()

plt.tight_layout()
plt.show()

* Campo de direcciones

Para cada punto $(t, T)$ se calcula la pendiente $dT/dt$ y se dibuja una flecha con esa dirección.
El color indica qué tan grande es la pendiente (rápido = rojo, lento = azul).

In [ ]:
t_g, T_g = np.meshgrid(np.linspace(0, t_final, 25), np.linspace(T_min_graf, T_max_graf, 25))

pendiente = -k * (T_g - T_amb)

# Se normaliza la longitud de las flechas (respetando la escala de los ejes)
u = np.ones_like(pendiente) / t_final
v = pendiente / (T_max_graf - T_min_graf)
norma = np.hypot(u, v)

# Se establece un valor acorde a la variación para los colores
C = np.abs(pendiente)

In [ ]:
plt.figure(figsize=(10, 5))

flechas = plt.quiver(t_g, T_g, u / norma, v / norma, C, angles="xy", cmap="coolwarm", width=0.003, zorder=0.1)
plt.colorbar(flechas, label="|dT/dt|")

plt.plot(t, T, "k", linewidth=2, label="T0 = 90")
plt.plot(t, T_2, "g", linewidth=2, label="T0 = 5")

plt.xlabel("t (min)")
plt.ylabel("Temperatura (°C)")
plt.title("Campo de direcciones — Modelo de temperatura")
plt.legend()
plt.grid()

plt.show()

* Línea de fase con muestras de la trayectoria

Como el modelo tiene una sola variable, el "retrato de fase" es la gráfica de $dT/dt$ contra $T$.
Se toman solo algunos puntos de la trayectoria (cada `paso` datos) para no saturar de flechas.

In [ ]:
paso = 25

dT_dt = -k * (T - T_amb)
dT_dt_2 = -k * (T_2 - T_amb)

# Se escogen cada 'paso' datos
xs, ys = T[::paso], dT_dt[::paso]
xs_2, ys_2 = T_2[::paso], dT_dt_2[::paso]

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(T, dT_dt, label="T0 = 90")
plt.plot(T_2, dT_dt_2, label="T0 = 5")
plt.plot(xs, ys, "o", color="tab:blue")
plt.plot(xs_2, ys_2, "o", color="tab:orange")
plt.axhline(0, color="black", linewidth=0.8)
plt.plot(T_amb, 0, "ro", label="Equilibrio")

plt.xlabel("T (°C)")
plt.ylabel("dT/dt")
plt.title("Línea de fase — Modelo de temperatura")
plt.legend()
plt.grid()

plt.show()

* Variaciones del modelo con Euler

Se repite el método para distintos valores de $k$, $T_a$ y $T_0$ usando una función auxiliar.

In [ ]:
def euler(f, T0, t):
    # Método de Euler para dT/dt = f(t, T) en los instantes del vector t
    T = np.zeros(len(t))
    T[0] = T0
    for n in range(len(t) - 1):
        T[n+1] = T[n] + (t[n+1] - t[n]) * f(t[n], T[n])
    return T

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4.8), sharey=True)

# (a) Variación de k
for k_i in k_lista:
    axs[0].plot(t, euler(lambda tt, TT: -k_i * (TT - T_amb), T_0, t), label=f"k = {k_i}")
axs[0].set_title("(a) Variación de k")

# (b) Variación de la temperatura ambiente
for Ta_i in Ta_lista:
    axs[1].plot(t, euler(lambda tt, TT: -k * (TT - Ta_i), T_0, t), label=f"Ta = {Ta_i}")
axs[1].set_title("(b) Variación de Ta")

# (c) Variación de la temperatura inicial
for T0_i in T0_lista:
    axs[2].plot(t, euler(lambda tt, TT: -k * (TT - T_amb), T0_i, t), label=f"T0 = {T0_i}")
axs[2].set_title("(c) Variación de T0")

for ax in axs:
    ax.set_xlabel("t (min)")
    ax.legend()
    ax.grid()
axs[0].set_ylabel("Temperatura (°C)")

plt.tight_layout()
plt.show()

* Efecto del tamaño del paso $h$ (estabilidad de Euler)

In [ ]:
t_final_2 = 20
t_exacto = np.linspace(0, t_final_2, 500)

plt.figure(figsize=(10, 5))
plt.plot(t_exacto, T_amb + (T_0 - T_amb) * np.exp(-k_rapido * t_exacto), "k", linewidth=2, label="Exacta")

for h_i in h_lista:
    t_i = np.arange(0, t_final_2 + h_i, h_i)
    T_i = euler(lambda tt, TT: -k_rapido * (TT - T_amb), T_0, t_i)
    plt.plot(t_i, T_i, "o-", markersize=4, label=f"h = {h_i}  (h·k = {h_i * k_rapido:.2f})")

plt.ylim(-100, 200)
plt.xlabel("t (min)")
plt.ylabel("Temperatura (°C)")
plt.title("Euler diverge cuando h·k > 2")
plt.legend()
plt.grid()

plt.show()

* Modelo modificado: temperatura ambiente variable

$T_a(t) = T_m + A\sin(2\pi t / P)$ con ciclo de 24 horas.

In [ ]:
T_m, A, P = 20.0, 8.0, 24.0
k_v = 0.5

t_v = np.arange(0, 72 + h, h)
T_ambiente = T_m + A * np.sin(2 * np.pi * t_v / P)

T_v = euler(lambda tt, TT: -k_v * (TT - (T_m + A * np.sin(2 * np.pi * tt / P))), 35.0, t_v)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(t_v, T_v, color="tab:red", label="Objeto T(t)")
plt.plot(t_v, T_ambiente, "--", color="gray", label="Ambiente Ta(t)")

plt.xlabel("t (h)")
plt.ylabel("Temperatura (°C)")
plt.title("Ambiente variable — Método de Euler")
plt.legend()
plt.grid()

plt.show()

* Punto de equilibrio

Se busca $-k(T - T_a) = 0$, es decir, $T^* = T_a$. Numéricamente, la trayectoria de Euler termina en ese valor:

In [ ]:
print(f"Temperatura final (T0 = 90): {T[-1]:.4f} °C")
print(f"Temperatura final (T0 = 5):  {T_2[-1]:.4f} °C")
print(f"Equilibrio teórico T* = T_amb = {T_amb} °C")
print(f"Autovalor: lambda = -k = {-k}  (negativo -> estable)")